Import required libraries

In [4]:
import os

Read the input dataset : Currently just a text file


In [8]:
with open("4_2_the-verdict.txt","r") as input_file:
    input_data = input_file.read()
input_file.close()

print("Number of characters ", len(input_data))

Number of characters  20479


Convert this text -> token set -> and assign each token with an embedding.

In [31]:
import re
scentence = "Hi, Hello this is dhiraj's laptop."


tokens = re.split(r'(\s)', scentence) # adding that () will also include the space as a token.
print(tokens[:20])

tokens = re.split(r'\s', scentence)
print(tokens[:20])

# r'\s' matches a single whitespace character (space, tab, newline, etc.).
# r'(\s)' matches a single whitespace character and captures it as a sub-group.

print(scentence.split(' ')[:20])

['Hi,', ' ', 'Hello', ' ', 'this', ' ', 'is', ' ', "dhiraj's", ' ', 'laptop.']
['Hi,', 'Hello', 'this', 'is', "dhiraj's", 'laptop.']
['Hi,', 'Hello', 'this', 'is', "dhiraj's", 'laptop.']


Break the input data set->to start tokenization.

In [36]:
import re

tokens = re.split(r'(\s)', scentence) # adding that () will also include the space as a token.
print(tokens[:20])

tokens = re.split(r'([,.]|\s)', scentence) # adding that () will also include the space as a token.
print(tokens[:20])

['Hi,', ' ', 'Hello', ' ', 'this', ' ', 'is', ' ', "dhiraj's", ' ', 'laptop.']
['Hi', ',', '', ' ', 'Hello', ' ', 'this', ' ', 'is', ' ', "dhiraj's", ' ', 'laptop', '.', '']


In [54]:
import re
tokens = re.split(r'([,.-]|\"|\s)', input_data) # adding that () will also include the space as a token.
print("orginal tokens", len(tokens ))
tokens = [ token for token in tokens if token.strip()] # token strip removes the space and if there is only space or empty -> it will be result in empty string.
print("After removing whitespaces/empty tokens", len(tokens ))
print(tokens[:20])

# removing whitespaces or not:
# - if we remove the whitespaces, we will lose the information about the structure of the text. For example, if we have a sentence "Hi Hello this is dhiraj's laptop", if we remove the whitespaces, we will get "HiHellothisisdhiraj'slaptop". This will make it difficult for the model to understand the structure of the text and to generate coherent responses.
# Similarly in coding tasks, removing whitespaces, might not be good idea. might disturb undersatnding of code.
# This might end up as a difference betwen good and great tokenizer.

orginal tokens 9051
After removing whitespaces/empty tokens 4562
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '-', '-', 'though', 'a', 'good', 'fellow', 'enough', '-', '-', 'so']


# Convert tokens to token IDs

In [56]:
vocab ={}
decoder = {}
print("Unique tokens", len(set(tokens)))
tokens = [ token.strip().lower() for token in tokens]
print("Unique tokens lower-cased", len(set(tokens)))
# set as same token with same token id, if it is repeated in the text.
for token_id, token in enumerate(set(tokens)):
        vocab[token] = token_id
        decoder[token_id] = token
print("Vocabulary size ", len(vocab), "decoder size ", len(decoder))
print("Sample vocab", vocab.items())

Unique tokens 1185
Unique tokens lower-cased 1185
Vocabulary size  1185 decoder size  1185
Sample vocab dict_items([('contended', 0), ('lit', 1), ('exterminating', 2), ('amplest', 3), ('mr', 4), ('sweetness', 5), ('strouds', 6), ('predicted', 7), ('consummate', 8), ('carlo', 9), ('by', 10), ('plain', 11), ('sight', 12), ('simpleton', 13), ('down', 14), ('idling', 15), ("money's", 16), ("wasn't", 17), ('twice', 18), ('protest', 19), ('stairs', 20), ('gone', 21), ('deprecating', 22), ('palm', 23), ('rickham!', 24), ('thin', 25), ('smile', 26), ('escape', 27), ("didn't", 28), ('immediately', 29), ('romantic!', 30), ('professional', 31), ('life', 32), ('note!', 33), ('reproduction', 34), ('question:', 35), ('go', 36), ('let', 37), ('curtains', 38), ("jack's", 39), ('pride', 40), ('very', 41), ('bits', 42), ("i'll", 43), ('forced', 44), ('pushed', 45), ('mantel', 46), ('suburban', 47), ('dubarry_', 48), ('given', 49), ('flowers', 50), ('worth', 51), ('when', 52), ('frame', 53), ('aside', 54

# Encode :convert text to token-ids
# Decode : Convert output token-Id to text

In [93]:
class Tokenizer:
    def __init__(self, vocab):
        self.token_to_id = { token:token_id for token, token_id in vocab.items()}
        self.id_to_token = { token_id:token for token, token_id in vocab.items()}
    def encode(self, input_text):
        tokens = re.split(r'([,.-]|\"|\'|\s)', input_text)
        tokens = [token.strip().lower() for token in tokens if token.strip()]
        token_ids = [ self.token_to_id[token] for token in tokens]
        return token_ids
    def decode(self, input_token_ids):
        tokens = [self.id_to_token[token_id] for token_id in input_token_ids]
        output = ' '.join(tokens)
        output = re.sub(r'\s+([,.-]|\"|\'|\s)',r'\1', output)
        return output

In [94]:
first_line_input_data ="I HAD always thought Jack Gisburn rather a cheap genius--though a good \
    fellow enough--so it was no great surprise to me to hear that,"


In [95]:
tokenizer = Tokenizer(vocab)
print("Input text ", first_line_input_data)
token_ids = tokenizer.encode(first_line_input_data)
print("Token ids ", token_ids)
output_text = tokenizer.decode(token_ids)
print( "Decoded text ", output_text)

Input text  I HAD always thought Jack Gisburn rather a cheap genius--though a good     fellow enough--so it was no great surprise to me to hear that,
Token ids  [471, 1111, 1127, 583, 825, 1036, 650, 921, 106, 151, 325, 325, 270, 921, 657, 581, 117, 325, 325, 155, 1058, 826, 988, 1129, 191, 428, 1072, 428, 309, 620, 989]
Decoded text  i had always thought jack gisburn rather a cheap genius-- though a good fellow enough-- so it was no great surprise to me to hear that,


What if the token is not in vocabulary?

In [96]:
first_line_input_data ="This is dhiraj's laptop. It is a good laptop. I like it."

tokenizer = Tokenizer(vocab)
print("Input text ", first_line_input_data)
token_ids = tokenizer.encode(first_line_input_data)
print("Token ids ", token_ids)
output_text = tokenizer.decode(token_ids)
print( "Decoded text ", output_text)

Input text  This is dhiraj's laptop. It is a good laptop. I like it.


KeyError: 'dhiraj'

Dhiraj is not handled.

In [127]:
class Tokenizer_special_tokens:
    def __init__(self, vocab):
        self.token_to_id = { token:token_id for token, token_id in vocab.items()}
        self.token_to_id['<unk>'] = len(vocab) # adding unknown token to handle out of vocabulary tokens.
        self.token_to_id['<eos>'] = len(vocab) + 1 # adding end of sentence token to handle end of sentence.
        self.id_to_token = { token_id:token for token, token_id in vocab.items()}
        self.id_to_token[len(vocab)] = '<unk>'
        self.id_to_token[len(vocab) + 1] = '<eos>'

    def encode(self, input_text):
        tokens = re.split(r'([,.-]|\"|\'|\s)', input_text)
        print(tokens)
        tokens = [token.strip().lower() for token in tokens if token.strip()]
        token_ids = [ self.token_to_id[token] if token in self.token_to_id.keys() else self.token_to_id['<unk>'] for token in tokens ]
        return token_ids
    
    def decode(self, input_token_ids):
        tokens = [self.id_to_token[token_id] for token_id in input_token_ids]
        output = ' '.join(tokens)
        output = re.sub(r'\s+([,.-]|\"|\'|\s)',r'\1', output)
        return output

In [128]:
first_line_input_data ="This is dhiraj's laptop. It is a good laptop. I like it. <EOS>"

tokenizer = Tokenizer_special_tokens(vocab)
print("Input text ", first_line_input_data)
token_ids = tokenizer.encode(first_line_input_data)
print("Token ids ", token_ids)
output_text = tokenizer.decode(token_ids)
print( "Decoded text ", output_text)

Input text  This is dhiraj's laptop. It is a good laptop. I like it. <EOS>
['This', ' ', 'is', ' ', 'dhiraj', "'", 's', ' ', 'laptop', '.', '', ' ', 'It', ' ', 'is', ' ', 'a', ' ', 'good', ' ', 'laptop', '.', '', ' ', 'I', ' ', 'like', ' ', 'it', '.', '', ' ', '<EOS>']
Token ids  [362, 1052, 1185, 310, 1185, 1185, 959, 1058, 1052, 921, 657, 1185, 959, 471, 545, 1058, 959, 1186]
Decoded text  this is <unk>' <unk> <unk>. it is a good <unk>. i like it. <eos>


# The unknown tokens are marked as <UNK> and also <EOS> is appropriately handled.

# Question: What happens when scentences of different lengths sent as part of the the batch.
# how its handled ? What are <PAD> tokens.


# IS junk token always required?
# Read about BPE tokenizer and how it handles out of vocabulary tokens : Again IS junk token always required?
